# Huấn luyện model phát hiện Chữ ký & Con dấu trên chứng từ (YOLOv8)

Notebook này chạy trên **Google Colab (GPU)**, thực hiện đầy đủ pipeline:
1. Sinh dữ liệu tổng hợp (synthetic) — không cần dataset có sẵn
2. Chia train/val
3. Train YOLOv8
4. Đánh giá riêng Precision/Recall cho từng lớp (signature vs stamp)
5. Thử inference trên ảnh mẫu
6. Tải model đã train về máy

**Trước khi chạy:** vào menu `Runtime > Change runtime type > GPU` (chọn T4 hoặc cao hơn) để có GPU miễn phí.


## 1. Cài đặt thư viện & kiểm tra GPU

In [ ]:
!pip install -q ultralytics
import torch
print("GPU kha dung:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Ten GPU:", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), "GB")
else:
    print("CANH BAO: Chua bat GPU. Vao Runtime > Change runtime type > chon GPU roi Restart runtime.")


## 2. (Tuỳ chọn) Kết nối Google Drive

Nên làm bước này để **lưu dataset và model đã train vào Drive**, tránh mất dữ liệu khi Colab bị ngắt kết nối (Colab free thường tự ngắt sau vài giờ hoặc khi hết idle time).


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = "/content/drive/MyDrive/signature_detection"
os.makedirs(PROJECT_DIR, exist_ok=True)
print("Thu muc luu du lieu/model:", PROJECT_DIR)


## 3. Ghi script sinh dữ liệu tổng hợp (`generate_dataset.py`)

Script tự sinh:
- Nền chứng từ giả (phiếu thu/chi, hoá đơn)
- Chữ ký (dùng chữ ký thật nếu bạn upload vào `signatures/`, hoặc tự sinh giả bằng Bezier nếu chưa có)
- Con dấu đỏ (sinh hoàn toàn bằng code)
- 4 kịch bản: rỗng / chỉ chữ ký / chữ ký+dấu tách rời / **chữ ký+dấu chồng lấn** (case khó, quan trọng)
- Augmentation mô phỏng scan thật (nghiêng, nhiễu, mờ, nén JPEG)


In [ ]:
%%writefile generate_dataset.py
"""
generate_dataset.py
====================
Sinh dữ liệu tổng hợp (synthetic) cho bài toán phát hiện CHỮ KÝ (signature) và
CON DẤU (stamp) trên ảnh chứng từ (phiếu thu/chi, hoá đơn), có chủ động tạo ra
các trường hợp con dấu đè lên chữ ký (occlusion) để model học tốt case khó này.

Output: ảnh .jpg + nhãn YOLO format (.txt) trong thư mục dataset/images, dataset/labels
Class id: 0 = signature, 1 = stamp

CÁCH DÙNG NHANH:
    python generate_dataset.py --num_samples 2000 --out_dir dataset

CHUẨN BỊ DỮ LIỆU ĐẦU VÀO (khuyến nghị để chất lượng thật hơn):
    - Tải chữ ký thật từ CEDAR / GPDS / ICDAR SigComp, để các ảnh .png/.jpg
      (nền trắng, mực đen/xanh) vào thư mục --signatures_dir
    - Nếu KHÔNG có, script tự sinh chữ ký giả bằng đường cong Bezier ngẫu nhiên
      (chất lượng thấp hơn thật nhưng vẫn giúp model học được hình dạng/pattern cơ bản)
    - Con dấu được sinh HOÀN TOÀN bằng code (không cần dataset), vì con dấu có
      cấu trúc hình học đều đặn (tròn/vuông + viền + chữ) dễ mô phỏng.
"""

import os
import random
import math
import argparse
import glob

import numpy as np
import cv2
from PIL import Image, ImageDraw, ImageFont, ImageFilter


# ----------------------------------------------------------------------------
# 1. SINH NỀN TÀI LIỆU (background giống phiếu thu/chi/hoá đơn)
# ----------------------------------------------------------------------------

def generate_background(width, height):
    """Sinh 1 ảnh nền trắng giống chứng từ: có bảng, đường kẻ, vài dòng text giả."""
    img = Image.new("RGB", (width, height), color=(255, 255, 255))
    draw = ImageDraw.Draw(img)

    try:
        font_title = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 22)
        font_text = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf", 14)
    except Exception:
        font_title = ImageFont.load_default()
        font_text = ImageFont.load_default()

    # Tiêu đề giả
    titles = ["PHIẾU CHI", "PHIẾU THU", "HOA DON GTGT", "BIEN BAN BAN GIAO", "PHIEU THANH TOAN"]
    draw.text((width * 0.32, 30), random.choice(titles), fill=(0, 0, 0), font=font_title)

    # Vài dòng thông tin giả phía trên
    fake_lines = [
        "So chung tu: {}".format(random.randint(1000, 9999)),
        "Ngay lap: {:02d}/{:02d}/2026".format(random.randint(1, 28), random.randint(1, 12)),
        "Don vi: Cong ty TNHH ABC",
        "Nguoi nop/nhan: Nguyen Van {}".format(random.choice("ABCDEFGH")),
        "So tien: {:,} VND".format(random.randint(100000, 50000000)),
    ]
    y = 80
    for line in fake_lines:
        draw.text((60, y), line, fill=(20, 20, 20), font=font_text)
        y += 26

    # Vẽ bảng kẻ ô giả (mô phỏng bảng chi tiết)
    table_top = y + 20
    table_bottom = int(height * 0.65)
    n_rows = random.randint(3, 6)
    row_h = (table_bottom - table_top) // n_rows
    for r in range(n_rows + 1):
        yy = table_top + r * row_h
        draw.line([(60, yy), (width - 60, yy)], fill=(0, 0, 0), width=1)
    for xx in [60, width * 0.6, width - 60]:
        draw.line([(xx, table_top), (xx, table_bottom)], fill=(0, 0, 0), width=1)

    # Khu vực chữ ký ở cuối trang (label gợi ý, không phải box nhãn)
    sign_zone_y = int(height * 0.78)
    draw.text((80, sign_zone_y), "Nguoi lap phieu", fill=(0, 0, 0), font=font_text)
    draw.text((width - 260, sign_zone_y), "Nguoi ky duyet", fill=(0, 0, 0), font=font_text)
    draw.text((80, sign_zone_y + 20), "(Ky, ghi ro ho ten)", fill=(90, 90, 90), font=font_text)
    draw.text((width - 260, sign_zone_y + 20), "(Ky, ghi ro ho ten)", fill=(90, 90, 90), font=font_text)

    return img


# ----------------------------------------------------------------------------
# 2. SINH CHỮ KÝ GIẢ (fallback khi chưa có dataset chữ ký thật)
# ----------------------------------------------------------------------------

def _bezier_point(p0, p1, p2, p3, t):
    x = (1 - t) ** 3 * p0[0] + 3 * (1 - t) ** 2 * t * p1[0] + 3 * (1 - t) * t ** 2 * p2[0] + t ** 3 * p3[0]
    y = (1 - t) ** 3 * p0[1] + 3 * (1 - t) ** 2 * t * p1[1] + 3 * (1 - t) * t ** 2 * p2[1] + t ** 3 * p3[1]
    return (x, y)


def generate_fake_signature(width=300, height=120, color=None):
    """
    Sinh chữ ký giả bằng nhiều đường cong Bezier nối tiếp nhau, mô phỏng nét bút.
    Chỉ nên dùng làm dữ liệu bổ sung/tạm thời -- ưu tiên dùng chữ ký thật (CEDAR/GPDS)
    nếu có, vì chữ ký thật có texture nét bút (độ đậm nhạt, tốc độ nét) mà cách sinh
    này không mô phỏng được đầy đủ.
    """
    img = Image.new("RGBA", (width, height), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)

    if color is None:
        # Màu mực phổ biến: xanh dương đậm hoặc đen
        color = random.choice([(20, 30, 120, 255), (10, 10, 10, 255), (0, 40, 90, 255)])

    n_strokes = random.randint(2, 4)
    cursor_x = width * 0.05
    for _ in range(n_strokes):
        stroke_len = random.uniform(width * 0.25, width * 0.5)
        p0 = (cursor_x, random.uniform(height * 0.3, height * 0.7))
        p1 = (cursor_x + stroke_len * 0.3, random.uniform(0, height))
        p2 = (cursor_x + stroke_len * 0.6, random.uniform(0, height))
        p3 = (cursor_x + stroke_len, random.uniform(height * 0.3, height * 0.7))

        pts = [_bezier_point(p0, p1, p2, p3, t) for t in np.linspace(0, 1, 60)]
        line_w = random.randint(2, 4)
        for i in range(len(pts) - 1):
            draw.line([pts[i], pts[i + 1]], fill=color, width=line_w)

        cursor_x += stroke_len * random.uniform(0.7, 0.95)

    # Vài nét gạch chân / dấu chấm ngẫu nhiên cho giống chữ ký thật hơn
    if random.random() < 0.5:
        yy = height * random.uniform(0.75, 0.9)
        draw.line([(width * 0.05, yy), (width * 0.7, yy)], fill=color, width=2)

    img = img.filter(ImageFilter.GaussianBlur(0.4))
    return img


def load_real_signatures(signatures_dir):
    """Load các ảnh chữ ký thật (đã tải sẵn từ CEDAR/GPDS...) và tách nền trắng -> alpha."""
    paths = glob.glob(os.path.join(signatures_dir, "*.png")) + \
        glob.glob(os.path.join(signatures_dir, "*.jpg")) + \
        glob.glob(os.path.join(signatures_dir, "*.jpeg"))
    signatures = []
    for p in paths:
        try:
            im = Image.open(p).convert("RGB")
            arr = np.array(im)
            # Coi pixel gần trắng là nền -> trong suốt
            gray = cv2.cvtColor(arr, cv2.COLOR_RGB2GRAY)
            alpha = np.where(gray > 235, 0, 255).astype(np.uint8)
            rgba = np.dstack([arr, alpha])
            signatures.append(Image.fromarray(rgba, mode="RGBA"))
        except Exception as e:
            print(f"[warn] Khong doc duoc {p}: {e}")
    return signatures


# ----------------------------------------------------------------------------
# 3. SINH CON DẤU (procedural, không cần dataset)
# ----------------------------------------------------------------------------

def generate_stamp(size=180):
    """Sinh con dấu tròn màu đỏ kiểu công ty/kho bạc, có viền + chữ vòng cung + ngôi sao/text giữa."""
    img = Image.new("RGBA", (size, size), (255, 255, 255, 0))
    draw = ImageDraw.Draw(img)
    red = (200, 20, 20, 200)  # màu mực dấu, hơi trong suốt như dấu thật đóng nhạt

    margin = 8
    draw.ellipse([margin, margin, size - margin, size - margin], outline=red, width=4)
    draw.ellipse([margin + 14, margin + 14, size - margin - 14, size - margin - 14], outline=red, width=2)

    try:
        font = ImageFont.truetype("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf", 14)
    except Exception:
        font = ImageFont.load_default()

    center = size / 2
    text_center = random.choice(["CTY ABC", "KE TOAN", "DA THU", "PHONG TC-KT"])
    bbox = draw.textbbox((0, 0), text_center, font=font)
    tw, th = bbox[2] - bbox[0], bbox[3] - bbox[1]
    draw.text((center - tw / 2, center - th / 2), text_center, fill=red, font=font)

    # Chữ cong theo viền trên (mô phỏng tên công ty chạy vòng cung)
    arc_text = "CONG TY TNHH THUONG MAI"
    radius = size / 2 - 26
    angle_start = 200
    angle_step = 10
    for i, ch in enumerate(arc_text[:16]):
        angle = math.radians(angle_start - i * angle_step)
        x = center + radius * math.cos(angle)
        y = center + radius * math.sin(angle)
        char_img = Image.new("RGBA", (20, 20), (255, 255, 255, 0))
        cd = ImageDraw.Draw(char_img)
        cd.text((2, 2), ch, fill=red, font=font)
        rot_angle = -(angle_start - i * angle_step) - 90
        char_img = char_img.rotate(rot_angle, expand=True)
        img.paste(char_img, (int(x - char_img.width / 2), int(y - char_img.height / 2)), char_img)

    # Làm mực hơi loang/nhạt không đều như dấu đóng tay thật
    arr = np.array(img)
    noise = np.random.normal(0, 12, arr[..., 3].shape).astype(np.int16)
    alpha = arr[..., 3].astype(np.int16) + noise
    arr[..., 3] = np.clip(alpha, 0, 255).astype(np.uint8)
    img = Image.fromarray(arr, mode="RGBA")
    img = img.rotate(random.uniform(-15, 15), expand=True)
    return img


# ----------------------------------------------------------------------------
# 4. GHÉP (COMPOSITE) SIGNATURE + STAMP LÊN NỀN, CÓ CHO PHÉP CHỒNG LẤN
# ----------------------------------------------------------------------------

def paste_with_alpha(bg, fg, x, y, opacity=1.0):
    """Dán ảnh fg (RGBA) lên bg tại (x,y) với độ mờ opacity, trả về bbox (x1,y1,x2,y2)."""
    fg = fg.copy()
    if opacity < 1.0:
        alpha = fg.split()[3].point(lambda p: int(p * opacity))
        fg.putalpha(alpha)
    bg.paste(fg, (x, y), fg)
    return (x, y, x + fg.width, y + fg.height)


def compose_sample(bg_size=(1000, 1400), signatures_pool=None):
    """
    Sinh 1 sample hoàn chỉnh: nền + (có thể có) chữ ký + (có thể có) con dấu,
    trả về ảnh PIL và list nhãn [(class_id, x1,y1,x2,y2), ...] theo pixel.

    class_id: 0 = signature, 1 = stamp
    """
    W, H = bg_size
    img = generate_background(W, H)
    labels = []

    # Vùng hợp lý để đặt chữ ký/dấu: 2 khu vực cuối trang (bên trái và bên phải)
    zones = [
        (60, int(H * 0.80), int(W * 0.42), int(H * 0.96)),
        (int(W * 0.55), int(H * 0.80), W - 60, int(H * 0.96)),
    ]

    # Quyết định kịch bản cho sample này
    r = random.random()
    if r < 0.10:
        scenario = "empty"          # không có gì (negative sample -- quan trọng để giảm false positive)
    elif r < 0.45:
        scenario = "sig_only"
    elif r < 0.70:
        scenario = "sig_and_stamp_separate"
    else:
        scenario = "sig_and_stamp_overlap"   # trường hợp khó: dấu đè lên chữ ký

    zone = random.choice(zones)
    zx1, zy1, zx2, zy2 = zone

    if scenario == "empty":
        pass

    else:
        # --- Lấy 1 chữ ký (thật nếu có, không thì sinh giả) ---
        if signatures_pool:
            sig = random.choice(signatures_pool).copy()
        else:
            sig = generate_fake_signature(
                width=random.randint(220, 320), height=random.randint(80, 130)
            )

        scale = random.uniform(0.8, 1.3)
        sig = sig.resize((int(sig.width * scale), int(sig.height * scale)))
        angle = random.uniform(-8, 8)
        sig = sig.rotate(angle, expand=True)

        max_x = max(zx1, zx2 - sig.width)
        max_y = max(zy1, zy2 - sig.height)
        sx = random.randint(zx1, max_x) if max_x > zx1 else zx1
        sy = random.randint(zy1, max_y) if max_y > zy1 else zy1

        sig_opacity = random.uniform(0.75, 1.0)
        sig_box = paste_with_alpha(img, sig, sx, sy, opacity=sig_opacity)
        labels.append((0,) + sig_box)  # class 0 = signature

        if scenario in ("sig_and_stamp_separate", "sig_and_stamp_overlap"):
            stamp = generate_stamp(size=random.randint(130, 190))
            stamp_opacity = random.uniform(0.55, 0.85)

            if scenario == "sig_and_stamp_overlap":
                # Cố tình đặt tâm con dấu lệch vào vùng chữ ký để tạo occlusion 30-70%
                overlap_ratio = random.uniform(0.3, 0.7)
                cx = int(sig_box[0] + (sig_box[2] - sig_box[0]) * overlap_ratio)
                cy = int((sig_box[1] + sig_box[3]) / 2)
                stx = cx - stamp.width // 2
                sty = cy - stamp.height // 2
            else:
                # Đặt cách xa chữ ký, không chồng lấn
                stx = sig_box[2] + random.randint(10, 40)
                sty = sig_box[1] - random.randint(0, 20)
                if stx + stamp.width > W - 20:
                    stx = max(20, sig_box[0] - stamp.width - 20)

            stx = max(0, min(stx, W - stamp.width))
            sty = max(0, min(sty, H - stamp.height))

            stamp_box = paste_with_alpha(img, stamp, stx, sty, opacity=stamp_opacity)
            labels.append((1,) + stamp_box)  # class 1 = stamp
            # Lưu ý quan trọng: box của signature GIỮ NGUYÊN như lúc dán,
            # KHÔNG cắt bớt phần bị dấu che -- vì ground truth phản ánh
            # vùng chữ ký thật sự tồn tại, kể cả khi bị che một phần.

    return img, labels


# ----------------------------------------------------------------------------
# 5. AUGMENTATION MÔ PHỎNG ẢNH SCAN THẬT
# ----------------------------------------------------------------------------

def apply_scan_augmentation(pil_img):
    """
    Mô phỏng artifact scan/photocopy không làm thay đổi hình học ảnh.

    Lưu ý: bbox đã được tạo trước khi gọi hàm này. Vì vậy không xoay/skew/crop
    toàn ảnh ở đây, nếu không nhãn YOLO sẽ bị lệch. Các augmentation hình học
    nên để Ultralytics xử lý trong train.py vì thư viện sẽ biến đổi bbox cùng ảnh.
    """
    img = np.array(pil_img.convert("RGB"))

    # 1. Nhiễu hạt kiểu scan
    if random.random() < 0.6:
        noise = np.random.normal(0, random.uniform(3, 10), img.shape).astype(np.int16)
        img = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)

    # 2. Làm mờ nhẹ (mô phỏng photocopy/out of focus)
    if random.random() < 0.4:
        k = random.choice([3, 5])
        img = cv2.GaussianBlur(img, (k, k), 0)

    # 3. Giảm tương phản nhẹ / ánh sáng không đều (mô phỏng scan ám vàng, thiếu sáng)
    if random.random() < 0.3:
        alpha = random.uniform(0.85, 1.05)
        beta = random.uniform(-10, 10)
        img = cv2.convertScaleAbs(img, alpha=alpha, beta=beta)

    out = Image.fromarray(img)

    # 4. Nén JPEG artifact (lưu rồi đọc lại với quality thấp)
    if random.random() < 0.5:
        import io
        buf = io.BytesIO()
        out.save(buf, format="JPEG", quality=random.randint(45, 80))
        buf.seek(0)
        out = Image.open(buf).convert("RGB")

    return out


# ----------------------------------------------------------------------------
# 6. XUẤT FORMAT YOLO
# ----------------------------------------------------------------------------

def save_yolo_label(labels, img_w, img_h, label_path):
    lines = []
    for cls, x1, y1, x2, y2 in labels:
        x1, x2 = max(0, x1), min(img_w, x2)
        y1, y2 = max(0, y1), min(img_h, y2)
        if x2 <= x1 or y2 <= y1:
            continue
        cx = (x1 + x2) / 2 / img_w
        cy = (y1 + y2) / 2 / img_h
        w = (x2 - x1) / img_w
        h = (y2 - y1) / img_h
        lines.append(f"{cls} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}")
    with open(label_path, "w") as f:
        f.write("\n".join(lines))


# ----------------------------------------------------------------------------
# 7. MAIN
# ----------------------------------------------------------------------------

def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--num_samples", type=int, default=200)
    parser.add_argument("--out_dir", type=str, default="dataset")
    parser.add_argument("--signatures_dir", type=str, default=None,
                         help="Thư mục chứa ảnh chữ ký thật đã tải (CEDAR/GPDS...). "
                              "Nếu không cung cấp, script sẽ tự sinh chữ ký giả.")
    parser.add_argument("--img_w", type=int, default=1000)
    parser.add_argument("--img_h", type=int, default=1400)
    args = parser.parse_args()

    img_dir = os.path.join(args.out_dir, "images")
    lbl_dir = os.path.join(args.out_dir, "labels")
    os.makedirs(img_dir, exist_ok=True)
    os.makedirs(lbl_dir, exist_ok=True)

    signatures_pool = None
    if args.signatures_dir and os.path.isdir(args.signatures_dir):
        signatures_pool = load_real_signatures(args.signatures_dir)
        print(f"Da load {len(signatures_pool)} chu ky that tu {args.signatures_dir}")
    else:
        print("Khong co --signatures_dir hop le -> se tu sinh chu ky gia (Bezier).")

    for i in range(args.num_samples):
        img, labels = compose_sample(bg_size=(args.img_w, args.img_h), signatures_pool=signatures_pool)
        img = apply_scan_augmentation(img)

        fname = f"sample_{i:05d}"
        img.save(os.path.join(img_dir, fname + ".jpg"), quality=90)
        save_yolo_label(labels, args.img_w, args.img_h, os.path.join(lbl_dir, fname + ".txt"))

        if (i + 1) % 100 == 0:
            print(f"Da sinh {i + 1}/{args.num_samples} samples")

    # Ghi file YAML tham khảo cho dataset CHƯA split. Không dùng file này để train
    # vì train/val sẽ trỏ cùng thư mục và làm metric validation bị ảo.
    yaml_path = os.path.join(args.out_dir, "data_unsplit_DO_NOT_TRAIN.yaml")
    with open(yaml_path, "w") as f:
        f.write(
            "# Dataset chua split: KHONG dung file nay de train/validate.\n"
            "# Hay chay split_dataset.py de tao data.yaml rieng cho train/val.\n"
            f"path: {os.path.abspath(args.out_dir)}\n"
            f"train: images\n"
            f"val: images\n"
            f"names:\n  0: signature\n  1: stamp\n"
        )

    print(f"\nHoan tat. Dataset tai: {os.path.abspath(args.out_dir)}")
    print(f"File tham khao dataset chua split: {yaml_path}")
    print("Buoc tiep theo nen lam:")
    print(f"  python split_dataset.py --src_dir {args.out_dir} --dst_dir {args.out_dir}_split --val_ratio 0.15")
    print("Sau do train voi file data.yaml da split, vi du:")
    print(f"  yolo detect train data={args.out_dir}_split/data.yaml model=yolov8n.pt imgsz=1024 epochs=50")


if __name__ == "__main__":
    main()


### (Tuỳ chọn) Upload chữ ký thật để tăng chất lượng dataset

Nếu bạn có ảnh chữ ký thật (ví dụ tải từ dataset CEDAR/GPDS, hoặc chữ ký nội bộ đã ẩn danh),
upload trực tiếp vào thư mục `signatures/` bằng ô bên dưới. Nếu bỏ qua bước này,
script sẽ tự sinh chữ ký giả (chất lượng thấp hơn thật nhưng vẫn train được).


In [ ]:
import os
os.makedirs("signatures", exist_ok=True)

from google.colab import files
print("Neu co anh chu ky that (.png/.jpg), chon file de upload. Bam Cancel de bo qua.")
try:
    uploaded = files.upload()
    for fname in uploaded.keys():
        os.rename(fname, os.path.join("signatures", fname))
    print(f"Da upload {len(uploaded)} anh chu ky vao thu muc signatures/")
except Exception as e:
    print("Bo qua upload chu ky that, se dung chu ky gia (Bezier).")


## 4. Sinh dataset

In [ ]:
NUM_SAMPLES = 3000   # tang len 5000-10000 neu can dataset lon hon, giam xuong ~500 de test nhanh
IMG_W, IMG_H = 1000, 1400

import os
SIG_DIR = "signatures" if len(os.listdir("signatures")) > 0 else None

if SIG_DIR:
    !python generate_dataset.py --num_samples {NUM_SAMPLES} --out_dir dataset --img_w {IMG_W} --img_h {IMG_H} --signatures_dir signatures
else:
    !python generate_dataset.py --num_samples {NUM_SAMPLES} --out_dir dataset --img_w {IMG_W} --img_h {IMG_H}


### Xem thử vài ảnh + bounding box để kiểm tra dữ liệu sinh ra có hợp lý không

In [ ]:
import cv2
import glob
import matplotlib.pyplot as plt

def draw_boxes(img_path, lbl_path):
    img = cv2.imread(img_path)
    h, w = img.shape[:2]
    colors = {0: (255, 0, 0), 1: (0, 0, 255)}   # signature=xanh, stamp=do (BGR)
    names = {0: "signature", 1: "stamp"}
    if os.path.exists(lbl_path):
        with open(lbl_path) as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                cls, cx, cy, bw, bh = map(float, line.split())
                cls = int(cls)
                x1 = int((cx - bw / 2) * w); y1 = int((cy - bh / 2) * h)
                x2 = int((cx + bw / 2) * w); y2 = int((cy + bh / 2) * h)
                cv2.rectangle(img, (x1, y1), (x2, y2), colors[cls], 3)
                cv2.putText(img, names[cls], (x1, max(0, y1 - 6)), cv2.FONT_HERSHEY_SIMPLEX, 0.7, colors[cls], 2)
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

sample_imgs = sorted(glob.glob("dataset/images/*.jpg"))[:4]
fig, axes = plt.subplots(1, len(sample_imgs), figsize=(20, 6))
for ax, img_path in zip(axes, sample_imgs):
    lbl_path = img_path.replace("images", "labels").replace(".jpg", ".txt")
    ax.imshow(draw_boxes(img_path, lbl_path))
    ax.axis("off")
plt.tight_layout()
plt.show()


## 5. Chia train/val

In [ ]:
%%writefile split_dataset.py
"""
split_dataset.py
==================
Chia dataset sinh ra từ generate_dataset.py (thư mục images/ + labels/ chung)
thành cấu trúc chuẩn YOLO: train/ và val/ riêng biệt, kèm data.yaml tương ứng.

CÁCH DÙNG:
    python split_dataset.py --src_dir dataset --dst_dir dataset_split --val_ratio 0.15
"""

import os
import shutil
import random
import argparse


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--src_dir", type=str, required=True,
                         help="Thư mục dataset gốc (có images/ và labels/ chung)")
    parser.add_argument("--dst_dir", type=str, required=True,
                         help="Thư mục đích, sẽ tạo train/ val/ bên trong")
    parser.add_argument("--val_ratio", type=float, default=0.15)
    parser.add_argument("--seed", type=int, default=42)
    args = parser.parse_args()

    if not 0 < args.val_ratio < 1:
        raise ValueError("--val_ratio phai nam trong khoang (0, 1)")

    random.seed(args.seed)

    img_dir = os.path.join(args.src_dir, "images")
    lbl_dir = os.path.join(args.src_dir, "labels")

    if not os.path.isdir(img_dir):
        raise FileNotFoundError(f"Khong tim thay thu muc images: {img_dir}")
    if not os.path.isdir(lbl_dir):
        raise FileNotFoundError(f"Khong tim thay thu muc labels: {lbl_dir}")

    files = [f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    if not files:
        raise ValueError(f"Khong co anh nao trong: {img_dir}")

    random.shuffle(files)

    n_val = max(1, int(len(files) * args.val_ratio))
    val_files = set(files[:n_val])
    train_files = files[n_val:]

    missing_labels = []
    for split_name, split_files in [("train", train_files), ("val", val_files)]:
        os.makedirs(os.path.join(args.dst_dir, split_name, "images"), exist_ok=True)
        os.makedirs(os.path.join(args.dst_dir, split_name, "labels"), exist_ok=True)
        for fname in split_files:
            base = os.path.splitext(fname)[0]
            shutil.copy(os.path.join(img_dir, fname),
                        os.path.join(args.dst_dir, split_name, "images", fname))
            lbl_src = os.path.join(lbl_dir, base + ".txt")
            lbl_dst = os.path.join(args.dst_dir, split_name, "labels", base + ".txt")
            if os.path.exists(lbl_src):
                shutil.copy(lbl_src, lbl_dst)
            else:
                missing_labels.append(fname)
                open(lbl_dst, "w").close()

    yaml_path = os.path.join(args.dst_dir, "data.yaml")
    with open(yaml_path, "w") as f:
        f.write(
            f"path: {os.path.abspath(args.dst_dir)}\n"
            f"train: train/images\n"
            f"val: val/images\n"
            f"names:\n  0: signature\n  1: stamp\n"
        )

    print(f"Train: {len(train_files)} anh | Val: {len(val_files)} anh")
    print(f"Da tao data.yaml tai: {yaml_path}")
    if missing_labels:
        print(f"[warn] Co {len(missing_labels)} anh thieu label; da tao file .txt rong.")


if __name__ == "__main__":
    main()


In [ ]:
!python split_dataset.py --src_dir dataset --dst_dir dataset_split --val_ratio 0.15

## 6. Train YOLOv8

Colab free thường cấp GPU **T4 (16GB VRAM)** — dư sức dùng `yolov8s.pt` với `imgsz=1024`, `batch=16`.
Nếu bạn thấy Colab gán GPU yếu hơn hoặc bị giới hạn VRAM, giảm `batch` hoặc `imgsz` theo gợi ý trong log lỗi.


In [ ]:
%%writefile train.py
"""
train.py
=========
Train model YOLOv8 phát hiện signature/stamp trên ảnh chứng từ.

Các lựa chọn cấu hình trong script này được chọn RIÊNG cho đặc thù bài toán:
  - imgsz=1024 (thay vi mac dinh 640): vi signature/stamp la object nho so voi
    ca trang tai lieu, giam resolution se lam mat chi tiet net chu ky mong.
  - model mac dinh yolov8s (khong phai n) de co du capacity phan biet net chu ky
    mong voi nen/duong ke bang; neu can nhe hon cho production co the đổi 'n'.
  - patience cao hon mac dinh vi dataset synthetic hoi "de" luc dau, tranh early
    stop qua som truoc khi model on dinh tren cac case kho (overlap).
  - augmentation hinh hoc (mosaic, flip) duoc GIAM bot so voi mac dinh vi:
      + KHONG nen flip ngang/doc: chu ky/dau bi lat se khong con giong chu ky/dau that
      + mosaic vua phai vi de sinh nhieu context nen khac nhau, nhung qua nhieu
        se pha vo vi tri "hop ly" cua chu ky (thuong o cuoi trang)

CACH DUNG:
    python train.py --data dataset_split/data.yaml --epochs 100 --imgsz 1024
"""

import argparse
from ultralytics import YOLO


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--data", type=str, required=True, help="Duong dan data.yaml")
    parser.add_argument("--model", type=str, default="yolov8s.pt",
                         help="Pretrained weight khoi tao (transfer learning). "
                              "Dung yolov8n.pt neu can nhe/nhanh hon cho production.")
    parser.add_argument("--epochs", type=int, default=100)
    parser.add_argument("--imgsz", type=int, default=1024)
    parser.add_argument("--batch", type=int, default=8)
    parser.add_argument("--patience", type=int, default=30)
    parser.add_argument("--project", type=str, default="runs_signature")
    parser.add_argument("--name", type=str, default="yolov8_sig_stamp")
    parser.add_argument("--device", type=str, default=None,
                         help="'0' = GPU dau tien, '0,1' = 2 GPU, 'cpu' = ep dung CPU. "
                              "Neu khong truyen, Ultralytics tu chon GPU neu co san.")
    parser.add_argument("--workers", type=int, default=8,
                         help="So luong dataloader worker. Tang len khi dung GPU de "
                              "tranh GPU bi 'doi' du lieu (bottleneck o CPU/disk).")
    args = parser.parse_args()

    model = YOLO(args.model)

    model.train(
        data=args.data,
        epochs=args.epochs,
        imgsz=args.imgsz,
        batch=args.batch,
        patience=args.patience,
        project=args.project,
        name=args.name,
        device=args.device,
        workers=args.workers,

        # --- Augmentation: tat cac phep bien doi khong hop ly voi chu ky/dau ---
        fliplr=0.0,      # KHONG lat ngang: chu ky bi lat khong con giong that
        flipud=0.0,      # KHONG lat doc
        mosaic=0.3,      # giam mosaic (mac dinh 1.0) de giu boi canh layout tai lieu
        mixup=0.0,       # tat mixup: tron 2 anh lam mo net chu ky, phan tac dung

        # --- Cac phep bien doi con lai giu muc vua phai, mo phong bien dang scan ---
        degrees=3.0,     # xoay nhe, giong nghieng trang khi scan
        translate=0.1,
        scale=0.3,
        shear=0.0,
        hsv_h=0.01, hsv_s=0.3, hsv_v=0.3,  # bien doi mau/anh sang vua phai

        # --- Ho tro object nho ---
        close_mosaic=10,  # tat mosaic o 10 epoch cuoi de model "quen" voi ty le that
    )

    # Chay validate cuoi cung, in ra metric tong quat
    metrics = model.val(data=args.data, imgsz=args.imgsz)
    print("\n=== KET QUA VALIDATION TONG QUAT ===")
    print(f"mAP50    : {metrics.box.map50:.4f}")
    print(f"mAP50-95 : {metrics.box.map:.4f}")
    print("\nChay them 'python eval_per_class.py' de xem chi tiet Precision/Recall "
          "TUNG LOP (signature vs stamp) -- quan trong de kiem tra xem signature "
          "co bi 'lep ve' so voi stamp hay khong.")


if __name__ == "__main__":
    main()


In [ ]:
!python train.py \
    --data dataset_split/data.yaml \
    --model yolov8s.pt \
    --epochs 100 \
    --imgsz 1024 \
    --batch 16 \
    --device 0 \
    --workers 8 \
    --project runs_signature \
    --name yolov8_sig_stamp


## 7. Đánh giá riêng Precision/Recall cho từng lớp (signature vs stamp)

In [ ]:
%%writefile eval_per_class.py
"""
eval_per_class.py
===================
Đánh giá model đã train, in ra Precision/Recall/mAP RIÊNG cho từng lớp
(signature vs stamp). Đây là bước quan trọng vì:

  - Metric tổng (mAP chung) có thể "che giấu" việc model học tốt stamp
    (dễ học vì hình dạng đều, màu đỏ nổi bật) nhưng học kém signature
    (khó học vì hình dạng bất định, dễ bị occlude bởi stamp).
  - Với bài toán nghiệp vụ kiểm tra chứng từ, RECALL của signature quan trọng
    hơn Precision (bỏ sót chữ ký nguy hiểm hơn báo nhầm) -- script này cảnh báo
    riêng nếu recall signature thấp hơn ngưỡng chấp nhận được.

CACH DUNG:
    python eval_per_class.py --weights runs_signature/yolov8_sig_stamp/weights/best.pt \
                              --data dataset_split/data.yaml
"""

import argparse
from ultralytics import YOLO

CLASS_NAMES = {0: "signature", 1: "stamp"}
MIN_ACCEPTABLE_RECALL_SIGNATURE = 0.85  # nguong nghiep vu de canh bao


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--weights", type=str, required=True, help="Duong dan best.pt")
    parser.add_argument("--data", type=str, required=True, help="Duong dan data.yaml")
    parser.add_argument("--imgsz", type=int, default=1024)
    parser.add_argument("--conf", type=float, default=0.25,
                         help="Nguong confidence de tinh Precision/Recall diem nay. "
                              "Neu uu tien recall cao (khuyen nghi cho bai toan nay), "
                              "co the giam xuong 0.1-0.15.")
    args = parser.parse_args()

    model = YOLO(args.weights)
    metrics = model.val(data=args.data, imgsz=args.imgsz, conf=args.conf)

    print("\n" + "=" * 60)
    print("KET QUA CHI TIET THEO TUNG LOP")
    print("=" * 60)

    # metrics.box.p, .r, .ap50, .ap la array theo thu tu class index
    p_per_class = metrics.box.p
    r_per_class = metrics.box.r
    ap50_per_class = metrics.box.ap50
    ap_per_class = metrics.box.ap

    warnings = []

    for idx, name in CLASS_NAMES.items():
        try:
            precision = p_per_class[idx]
            recall = r_per_class[idx]
            ap50 = ap50_per_class[idx]
            ap = ap_per_class[idx]
        except (IndexError, TypeError):
            print(f"[{name}] Khong co du lieu (co the khong co sample nao trong val set)")
            continue

        print(f"\nLop: {name}")
        print(f"  Precision : {precision:.4f}")
        print(f"  Recall    : {recall:.4f}")
        print(f"  mAP50     : {ap50:.4f}")
        print(f"  mAP50-95  : {ap:.4f}")

        if name == "signature" and recall < MIN_ACCEPTABLE_RECALL_SIGNATURE:
            warnings.append(
                f"[CANH BAO] Recall cua 'signature' = {recall:.4f}, "
                f"THAP hon nguong nghiep vu ({MIN_ACCEPTABLE_RECALL_SIGNATURE}). "
                f"Model dang co nguy co BO SOT chu ky -- rui ro cao cho nghiep vu "
                f"kiem tra chung tu. Goi y khac phuc:\n"
                f"    1. Giam --conf khi inference (vd 0.1-0.15) de tang recall\n"
                f"    2. Them nhieu sample 'signature bi stamp che' vao training set\n"
                f"    3. Tang trong so loss cho class signature (cls weight)\n"
                f"    4. Kiem tra lai anchor/resolution -- co the object qua nho"
            )

    # So sanh cheo: neu stamp recall vuot troi han signature -> dau hieu mat can bang
    try:
        r_sig = r_per_class[0]
        r_stamp = r_per_class[1]
        if r_stamp - r_sig > 0.15:
            warnings.append(
                f"[CANH BAO] Chenh lech recall giua stamp ({r_stamp:.4f}) va "
                f"signature ({r_sig:.4f}) qua lon (>0.15). Day la dau hieu dien hinh "
                f"cua viec model 'thien vi' hoc dac trung de (dau mau do, hinh tron deu) "
                f"va bo qua dac trung kho hon (net chu ky mong, bat dinh hinh) -- dung "
                f"nhu van de occlusion da trao doi truoc do."
            )
    except (IndexError, TypeError):
        pass

    if warnings:
        print("\n" + "=" * 60)
        print("CAC CANH BAO NGHIEP VU")
        print("=" * 60)
        for w in warnings:
            print("\n" + w)
    else:
        print("\nKhong co canh bao -- Recall cua signature dat nguong chap nhan duoc.")


if __name__ == "__main__":
    main()


In [ ]:
!python eval_per_class.py \
    --weights runs_signature/yolov8_sig_stamp/weights/best.pt \
    --data dataset_split/data.yaml \
    --imgsz 1024 \
    --conf 0.25


## 8. Thử inference trên vài ảnh validation

In [ ]:
from ultralytics import YOLO
import matplotlib.pyplot as plt

model = YOLO("runs_signature/yolov8_sig_stamp/weights/best.pt")

val_imgs = sorted(glob.glob("dataset_split/val/images/*.jpg"))[:4]
results = model.predict(val_imgs, conf=0.25, imgsz=1024)

fig, axes = plt.subplots(1, len(results), figsize=(20, 6))
for ax, r in zip(axes, results):
    ax.imshow(cv2.cvtColor(r.plot(), cv2.COLOR_BGR2RGB))
    ax.axis("off")
plt.tight_layout()
plt.show()


## 9. Lưu model về Google Drive / tải trực tiếp về máy

Chạy 1 trong 2 cách bên dưới (hoặc cả hai).


In [ ]:
# Cach 1: Copy vao Google Drive (neu da mount o Buoc 2)
import shutil
best_weight = "runs_signature/yolov8_sig_stamp/weights/best.pt"
dst = os.path.join(PROJECT_DIR, "best.pt")
shutil.copy(best_weight, dst)
print("Da luu model vao:", dst)


In [ ]:
# Cach 2: Tai truc tiep ve may
from google.colab import files
files.download("runs_signature/yolov8_sig_stamp/weights/best.pt")


## Ghi chú quan trọng

- **Nếu recall của `signature` thấp** (xem cảnh báo ở Bước 7): thử giảm `--conf` khi inference (0.1–0.15), hoặc tăng `NUM_SAMPLES` ở Bước 4, đặc biệt tăng tỷ lệ case chồng lấn (đã có sẵn trong `generate_dataset.py`, chiếm ~30% dataset mặc định).
- **Nếu gặp lỗi CUDA out of memory:** giảm `--batch` xuống 8 hoặc `--imgsz` xuống 640-800 ở Bước 6.
- **Dữ liệu thật luôn tốt hơn synthetic:** khi có ảnh chứng từ thật (đã ẩn danh), hãy trộn vào `dataset/images` + `dataset/labels` (đúng format YOLO) trước Bước 5, rồi chạy lại pipeline từ đó.
- Colab free có thể tự ngắt kết nối sau vài giờ — nhớ đã mount Drive (Bước 2) để không mất kết quả.
